In [0]:
%run ../00_common/data_utils

In [0]:
def update_delete_action_records_with_clearpii(task_id):
    """
    按DELETE动作清空PII字段，并将命中记录的master consumer标记为当前task_id。

    实现要求（与参考SQL对齐）：
    1. emedia: 清空scme_address，并将scme_sourcetimestamp更新为srcc_sourcetimestamp
    2. phone: 清空scph_phonenumber，并将scph_sourcetimestamp更新为srcc_sourcetimestamp
    3. address: 清空scad_address1/2/3，并将scad_sourcetimestamp更新为srcc_sourcetimestamp
    4. optin: 将scop_optin_flag置0，并将scop_optin_dt更新为srcc_sourcetimestamp
    5. 将原SQL中scon_cbr_flag = 0替换为：
       - t_master_consumer.task_id = 当前task_id
       - scon_update_dt = current_timestamp
       - scon_update_uid = 'ELC'

    注意：所有Market统一逻辑处理，不做市场分支。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    silver_db = get_env_config('silver_consumer_cleansed_database')

    master_consumer_table_name = f"{golden_db}.t_master_consumer"
    master_emedia_table_name = f"{golden_db}.t_master_emedia"
    master_phone_table_name = f"{golden_db}.t_master_phone"
    master_address_table_name = f"{golden_db}.t_master_address"
    master_optin_table_name = f"{golden_db}.t_master_optin"

    clean_consumer_df = spark.table(f"{silver_db}.t_clean_consumer").where(f"TASK_ID = '{task_id}'")
    master_consumer_df = spark.table(master_consumer_table_name)
    master_emedia_df = spark.table(master_emedia_table_name)
    master_phone_df = spark.table(master_phone_table_name)
    master_address_df = spark.table(master_address_table_name)
    master_optin_df = spark.table(master_optin_table_name)

    # 先构造DELETE来源与master_consumer的关联（业务键 + timestamp条件）
    delete_consumer_base_df = (
        master_consumer_df.alias("scon")
        .join(
            clean_consumer_df.alias("srcc"),
            (F.col("scon.scon_srcs_code") == F.col("srcc.srcc_srcs_code")) &
            (F.col("scon.scon_mrkt_code") == F.col("srcc.srcc_mrkt_code")) &
            (F.col("scon.scon_brnd_code") == F.col("srcc.srcc_brnd_code")) &
            (F.col("scon.scon_consumerid") == F.col("srcc.srcc_consumerid")),
            "inner"
        )
        .where(
            (F.upper(F.col("srcc.srcc_action")) == "DELETE") &
            (F.col("srcc.srcc_sourcetimestamp") >= F.col("scon.scon_sourcetimestamp"))
        )
        .select(
            F.col("scon.scon_id"),
            F.col("scon.scon_mrkt_code"),
            F.col("srcc.srcc_sourcetimestamp")
        )
        .cache()
    )
    delete_consumer_base_count = delete_consumer_base_df.count()

    if delete_consumer_base_count == 0:
        print("DELETE clear-PII completed: emedia=0, phone=0, address=0, optin=0, consumer_task_updated=0")
        delete_consumer_base_df.unpersist()
        return

    # 1) 清空emedia地址（仅更新原本非空地址）
    emedia_to_update = (
        master_emedia_df.alias("scme")
        .join(
            delete_consumer_base_df.alias("base"),
            (F.col("scme.scme_scon_id") == F.col("base.scon_id")) &
            (F.col("scme.scme_mrkt_code") == F.col("base.scon_mrkt_code")),
            "inner"
        )
        .where(F.coalesce(F.col("scme.scme_address"), F.lit("")) != "")
        .groupBy("scme.scme_id", "scme.scme_mrkt_code")
        .agg(F.max("base.srcc_sourcetimestamp").alias("delete_sourcetimestamp"))
        .cache()
    )

    emedia_update_count = emedia_to_update.count()
    if emedia_update_count > 0:
        DeltaTable.forName(spark, master_emedia_table_name).alias("target").merge(
            emedia_to_update.alias("source"),
            """
            target.scme_id = source.scme_id AND
            target.scme_mrkt_code = source.scme_mrkt_code
            """
        ).whenMatchedUpdate(
            set={
                "scme_address": F.lit(""),
                "scme_sourcetimestamp": F.col("source.delete_sourcetimestamp"),
                "scme_update_dt": F.current_timestamp(),
                "scme_update_uid": F.lit("ELC")
            }
        ).execute()

    # 2) 清空phone号码（仅更新原本非空号码）
    phone_to_update = (
        master_phone_df.alias("scph")
        .join(
            delete_consumer_base_df.alias("base"),
            (F.col("scph.scph_scon_id") == F.col("base.scon_id")) &
            (F.col("scph.scph_mrkt_code") == F.col("base.scon_mrkt_code")),
            "inner"
        )
        .where(F.coalesce(F.col("scph.scph_phonenumber"), F.lit("")) != "")
        .groupBy("scph.scph_id", "scph.scph_mrkt_code")
        .agg(F.max("base.srcc_sourcetimestamp").alias("delete_sourcetimestamp"))
        .cache()
    )

    phone_update_count = phone_to_update.count()
    if phone_update_count > 0:
        DeltaTable.forName(spark, master_phone_table_name).alias("target").merge(
            phone_to_update.alias("source"),
            """
            target.scph_id = source.scph_id AND
            target.scph_mrkt_code = source.scph_mrkt_code
            """
        ).whenMatchedUpdate(
            set={
                "scph_phonenumber": F.lit(""),
                "scph_sourcetimestamp": F.col("source.delete_sourcetimestamp"),
                "scph_update_dt": F.current_timestamp(),
                "scph_update_uid": F.lit("ELC")
            }
        ).execute()

    # 3) 清空address1/2/3（严格按SQL逻辑，不额外判断原值是否为空）
    address_to_update = (
        master_address_df.alias("scad")
        .join(
            delete_consumer_base_df.alias("base"),
            (F.col("scad.scad_scon_id") == F.col("base.scon_id")) &
            (F.col("scad.scad_mrkt_code") == F.col("base.scon_mrkt_code")),
            "inner"
        )
        .groupBy("scad.scad_id", "scad.scad_mrkt_code")
        .agg(F.max("base.srcc_sourcetimestamp").alias("delete_sourcetimestamp"))
        .cache()
    )

    address_update_count = address_to_update.count()
    if address_update_count > 0:
        DeltaTable.forName(spark, master_address_table_name).alias("target").merge(
            address_to_update.alias("source"),
            """
            target.scad_id = source.scad_id AND
            target.scad_mrkt_code = source.scad_mrkt_code
            """
        ).whenMatchedUpdate(
            set={
                "scad_address1": F.lit(""),
                "scad_address2": F.lit(""),
                "scad_address3": F.lit(""),
                "scad_sourcetimestamp": F.col("source.delete_sourcetimestamp"),
                "scad_update_dt": F.current_timestamp(),
                "scad_update_uid": F.lit("ELC")
            }
        ).execute()

    # 4) 更新optin（仅更新原本optin_flag非0记录）
    optin_to_update = (
        master_optin_df.alias("scop")
        .join(
            delete_consumer_base_df.alias("base"),
            (F.col("scop.scop_scon_id") == F.col("base.scon_id")) &
            (F.col("scop.scop_mrkt_code") == F.col("base.scon_mrkt_code")),
            "inner"
        )
        .where(F.coalesce(F.col("scop.scop_optin_flag").cast("String"), F.lit("x")) != "x")
        .groupBy("scop.scop_id", "scop.scop_mrkt_code")
        .agg(F.max("base.srcc_sourcetimestamp").alias("delete_sourcetimestamp"))
        .cache()
    )

    optin_update_count = optin_to_update.count()
    if optin_update_count > 0:
        DeltaTable.forName(spark, master_optin_table_name).alias("target").merge(
            optin_to_update.alias("source"),
            """
            target.scop_id = source.scop_id AND
            target.scop_mrkt_code = source.scop_mrkt_code
            """
        ).whenMatchedUpdate(
            set={
                "scop_optin_flag": F.lit(False),
                "scop_optin_dt": F.col("source.delete_sourcetimestamp"),
                "scop_update_dt": F.current_timestamp(),
                "scop_update_uid": F.lit("ELC")
            }
        ).execute()

    # 5) 将命中的master consumer记录更新为当前task（替代scon_cbr_flag=0）
    consumers_to_update_task = (
        emedia_to_update.select(
            F.col("scme_id").alias("related_id"),
            F.col("scme_mrkt_code").alias("mrkt_code")
        )
        .join(
            master_emedia_df.alias("scme"),
            (F.col("related_id") == F.col("scme.scme_id")) &
            (F.col("mrkt_code") == F.col("scme.scme_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scme.scme_scon_id").alias("scon_id"),
            F.col("scme.scme_mrkt_code").alias("scon_mrkt_code")
        )
        .union(
            phone_to_update.select(
                F.col("scph_id").alias("related_id"),
                F.col("scph_mrkt_code").alias("mrkt_code")
            )
            .join(
                master_phone_df.alias("scph"),
                (F.col("related_id") == F.col("scph.scph_id")) &
                (F.col("mrkt_code") == F.col("scph.scph_mrkt_code")),
                "inner"
            )
            .select(
                F.col("scph.scph_scon_id").alias("scon_id"),
                F.col("scph.scph_mrkt_code").alias("scon_mrkt_code")
            )
        )
        .union(
            address_to_update.select(
                F.col("scad_id").alias("related_id"),
                F.col("scad_mrkt_code").alias("mrkt_code")
            )
            .join(
                master_address_df.alias("scad"),
                (F.col("related_id") == F.col("scad.scad_id")) &
                (F.col("mrkt_code") == F.col("scad.scad_mrkt_code")),
                "inner"
            )
            .select(
                F.col("scad.scad_scon_id").alias("scon_id"),
                F.col("scad.scad_mrkt_code").alias("scon_mrkt_code")
            )
        )
        .union(
            optin_to_update.select(
                F.col("scop_id").alias("related_id"),
                F.col("scop_mrkt_code").alias("mrkt_code")
            )
            .join(
                master_optin_df.alias("scop"),
                (F.col("related_id") == F.col("scop.scop_id")) &
                (F.col("mrkt_code") == F.col("scop.scop_mrkt_code")),
                "inner"
            )
            .select(
                F.col("scop.scop_scon_id").alias("scon_id"),
                F.col("scop.scop_mrkt_code").alias("scon_mrkt_code")
            )
        )
        .distinct()
        .cache()
    )

    consumer_update_count = consumers_to_update_task.count()
    if consumer_update_count > 0:
        DeltaTable.forName(spark, master_consumer_table_name).alias("target").merge(
            consumers_to_update_task.alias("source"),
            """
            target.scon_id = source.scon_id AND
            target.scon_mrkt_code = source.scon_mrkt_code
            """
        ).whenMatchedUpdate(
            set={
                "task_id": F.lit(task_id),
                "scon_update_dt": F.current_timestamp(),
                "scon_update_uid": F.lit("ELC")
            }
        ).execute()

    print(
        f"DELETE clear-PII completed: emedia={emedia_update_count}, "
        f"phone={phone_update_count}, address={address_update_count}, "
        f"optin={optin_update_count}, consumer_task_updated={consumer_update_count}"
    )

    consumers_to_update_task.unpersist()
    emedia_to_update.unpersist()
    phone_to_update.unpersist()
    address_to_update.unpersist()
    optin_to_update.unpersist()
    delete_consumer_base_df.unpersist()

In [0]:
def update_acs_optin_task_id(task_id):
    """
    更新ACS系统相关的consumer记录的task_id，标记为需要重新处理
    
    逻辑说明：
    1. 查找在排除系统（如ACS）中存在的email/phone联系方式
    2. 在主数据中查找拥有这些联系方式的consumer
    3. 将这些consumer的task_id设置为当前任务ID，标记为需要重新进行处理
    
    兼容所有market的逻辑：
    - 支持从dim_excludesource表读取排除系统的source code（支持多个）
    - 支持从t_merge_exclude_config表读取每个market的硬编码排除值
    - 分别处理emedia(email)和phone两种联系方式
    
    Args:
        task_id: 当前任务ID
    """
    master_consumer_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer"
    cleansed_database = get_env_config('silver_consumer_cleansed_database')
    config_database = get_env_config('config_database')
    
    # 读取配置表
    exclude_source_df = spark.table(f"{config_database}.t_merge_exclude_consumer_config").filter(F.col("tmec_type") == 'ACS')  # ACS等排除系统的source codes
    exclude_config_df = spark.table(f"{config_database}.t_merge_exclude_config")  # 硬编码排除值（email/phone）
    
    # 读取数据表
    clean_consumer_df = spark.table(f"{cleansed_database}.t_clean_consumer").where(f"TASK_ID = '{task_id}'").filter(F.col("IS_INCLUDE") == True)
    clean_emedia_df = spark.table(f"{cleansed_database}.t_clean_emedia").where(f"TASK_ID = '{task_id}'")
    clean_phone_df = spark.table(f"{cleansed_database}.t_clean_phone").where(f"TASK_ID = '{task_id}'")
    master_consumer_df = spark.table(master_consumer_table_name)
    master_emedia_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_emedia")
    master_phone_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_phone")
    
    # Part 1: 查找在排除系统中的emedia地址，并找到主数据中拥有这些地址的consumer
    # 获取每个market的email排除值
    email_exclude_values = (
        exclude_config_df
        .where("exclude_type = 'mediaAddress'")
        .select("MarketCode", "exclude_value")
        .groupBy("MarketCode")
        .agg(F.collect_set("exclude_value").alias("excluded_emails"))
    )
    
    # 从排除系统（如ACS）中获取email地址
    acs_emails_df = (
        clean_emedia_df.alias("srce")
        .join(
            clean_consumer_df.alias("srcc"),
            (F.col("srce.srce_srcc_id") == F.col("srcc.srcc_id")) &
            (F.col("srce.srce_mrkt_code") == F.col("srcc.srcc_mrkt_code")),
            "inner"
        )
        .join(
            exclude_source_df.alias("es"),
            (F.col("srcc.srcc_srcs_code") == F.col("es.tmec_sourcesystemcode")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("es.tmec_marketcode")),
            "inner"
        )
        .where("srce.srce_address IS NOT NULL AND srce.srce_address != ''")
        .select(
            F.col("srcc.srcc_mrkt_code").alias("market_code"),
            F.col("srce.srce_address").alias("email_address")
        )
        .distinct()
    )
    
    # 关联排除值并过滤
    acs_emails_filtered = (
        acs_emails_df.alias("ae")
        .join(
            email_exclude_values.alias("ev"),
            F.col("ae.market_code") == F.col("ev.MarketCode"),
            "left"
        )
        # 使用array_contains检查email是否在排除列表中，如果不在排除列表中则保留
        .where(
            F.col("ev.excluded_emails").isNull() | 
            ~F.array_contains(F.col("ev.excluded_emails"), F.col("ae.email_address"))
        )
        .select("ae.market_code", "ae.email_address")
    )
    
    # 在主数据中查找拥有这些email的consumer
    scon_ids_from_email = (
        master_consumer_df.alias("scon")
        .join(
            master_emedia_df.alias("scme"),
            (F.col("scon.scon_id") == F.col("scme.scme_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scme.scme_mrkt_code")),
            "inner"
        )
        .join(
            acs_emails_filtered.alias("acs_em"),
            (F.col("scme.scme_address") == F.col("acs_em.email_address")) &
            (F.col("scon.scon_mrkt_code") == F.col("acs_em.market_code")),
            "inner"
        )
        .select(F.col("scon.scon_id"), F.col("scon.scon_mrkt_code"))
        .distinct()
    )
    
    # Part 2: 查找在排除系统中的phone号码，并找到主数据中拥有这些号码的consumer
    # 获取每个market的phone排除值
    phone_exclude_values = (
        exclude_config_df
        .where("exclude_type = 'phoneNumber'")
        .select("MarketCode", "exclude_value")
        .groupBy("MarketCode")
        .agg(F.collect_set("exclude_value").alias("excluded_phones"))
    )
    
    # 从排除系统（如ACS）中获取phone号码
    acs_phones_df = (
        clean_phone_df.alias("srcp")
        .join(
            clean_consumer_df.alias("srcc"),
            (F.col("srcp.srcp_srcc_id") == F.col("srcc.srcc_id")) &
            (F.col("srcp.srcp_mrkt_code") == F.col("srcc.srcc_mrkt_code")),
            "inner"
        )
        .join(
            exclude_source_df.alias("es"),
            (F.col("srcc.srcc_srcs_code") == F.col("es.tmec_sourcesystemcode")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("es.tmec_marketcode")),
            "inner"
        )
        .where("srcp.srcp_phonenumber IS NOT NULL AND srcp.srcp_phonenumber != ''")
        .select(
            F.col("srcc.srcc_mrkt_code").alias("market_code"),
            F.col("srcp.srcp_phonenumber").alias("phone_number")
        )
        .distinct()
    )
    
    # 关联排除值并过滤
    acs_phones_filtered = (
        acs_phones_df.alias("ap")
        .join(
            phone_exclude_values.alias("pv"),
            F.col("ap.market_code") == F.col("pv.MarketCode"),
            "left"
        )
        # 使用array_contains检查phone是否在排除列表中，如果不在排除列表中则保留
        .where(
            F.col("pv.excluded_phones").isNull() | 
            ~F.array_contains(F.col("pv.excluded_phones"), F.col("ap.phone_number"))
        )
        .select("ap.market_code", "ap.phone_number")
    )
    
    # 在主数据中查找拥有这些phone的consumer
    scon_ids_from_phone = (
        master_consumer_df.alias("scon")
        .join(
            master_phone_df.alias("scph"),
            (F.col("scon.scon_id") == F.col("scph.scph_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scph.scph_mrkt_code")),
            "inner"
        )
        .join(
            acs_phones_filtered.alias("acs_ph"),
            (F.col("scph.scph_phonenumber") == F.col("acs_ph.phone_number")) &
            (F.col("scon.scon_mrkt_code") == F.col("acs_ph.market_code")),
            "inner"
        )
        .select(F.col("scon.scon_id"), F.col("scon.scon_mrkt_code"))
        .distinct()
    )
    
    # 合并email和phone的结果
    all_scon_ids_to_update = scon_ids_from_email.union(scon_ids_from_phone).distinct()
    
    # 更新sconsumer表的task_id
    if all_scon_ids_to_update.count() > 0:
        master_consumer_delta = DeltaTable.forName(spark, master_consumer_table_name)
        
        master_consumer_delta.alias("target").merge(
            all_scon_ids_to_update.alias("source"),
            """
            target.scon_id = source.scon_id AND 
            target.scon_mrkt_code = source.scon_mrkt_code
            """
        ).whenMatchedUpdate(
            set = {
                "task_id": F.lit(task_id),
                "scon_update_dt": F.current_timestamp(),
                "scon_update_uid": F.lit("ELC")
            }
        ).execute()
        
        print(f"Updated {all_scon_ids_to_update.count()} consumer records' task_id for ACS optin list")
    else:
        print("No consumer records to update for ACS optin list")

In [0]:
def calc_consumer_master(task_id):
    master_consumer_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer"
    config_table_name = f"{get_env_config('config_database')}.t_merge_exclude_consumer_config"
    
    # 1. 初始化数据源
    itermediate_consumer_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer") \
        .where(f"TASK_ID = '{task_id}'")\
        .filter(F.col("IS_INCLUDE") == True) 
    
    itermediate_matchgroup_df = get_ukey_group_by_process(task_id)

    itermediate_cidgroup_df = get_cidGroup_by_process(task_id) \
        .where(f"TASK_ID = '{task_id}' AND record_type = '{CID_MATCH_RECORD_TYPE_BATCH}'") \
        .select("mrkt_code", "srcc_id", "new_mapping_conusmer_id") \
        .distinct()

    master_consumer_df = spark.table(master_consumer_table_name)
    
    # 2. 处理Consumer数据（兼容所有Market的通用逻辑）
    # 读取排除系统配置表，获取ACS和LineBind的source system codes
    exclude_source_df = spark.table(config_table_name) \
        .where("tmec_type in ('ACS', 'LineBind')") \
        .select("tmec_sourcesystemcode", "tmec_marketcode") \
        .distinct()
    
    # Part 1: 处理常规记录（NOT IN ACS/LineBind source codes）
    # 这部分有DELETE字段清空逻辑，有timestamp比较
    consumer_part1_df = (
        itermediate_consumer_df.alias("ssc")
        .join(
            itermediate_matchgroup_df.alias("tmg"),
            (F.col("ssc.SRCC_ID") == F.col("tmg.srcc_id")) &
            (F.col("ssc.srcc_mrkt_code") == F.col("tmg.mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("sc"),
            (F.col("ssc.srcc_srcs_code") == F.col("sc.scon_srcs_code")) &
            (F.col("ssc.srcc_mrkt_code") == F.col("sc.scon_mrkt_code")) &
            (F.col("ssc.srcc_brnd_code") == F.col("sc.scon_brnd_code")) &
            (F.col("ssc.srcc_consumerid") == F.col("sc.scon_consumerid")),
            "left"
        )
        .where(
            F.col("ssc.SRCC_SOURCETIMESTAMP") >= 
            F.coalesce(F.col("sc.scon_sourcetimestamp"), F.lit("1900-01-01"))
        )
        .where(
            # 所有市场统一过滤UNBIND动作
            ~(F.upper(F.col("ssc.SRCC_ACTION")) == "UNBIND")
        )
        .join(
            exclude_source_df.alias("es"),
            (F.col("ssc.srcc_srcs_code") == F.col("es.tmec_sourcesystemcode")) &
            (F.col("ssc.srcc_mrkt_code") == F.col("es.tmec_marketcode")),
            "left_anti"  # 使用left anti join排除在ACS/LineBind中的记录
        )
        .select(
            "ssc.SRCC_ID",
            "ssc.SRCC_ACTION",
            "ssc.SRCC_SRCS_CODE",
            "ssc.SRCC_SOURCETIMESTAMP",
            "ssc.SRCC_MRKT_CODE",
            "ssc.SRCC_AFF_CODE",
            "ssc.SRCC_DVSN_CODE",
            "ssc.SRCC_BRND_CODE",
            "ssc.SRCC_CONSUMERID",
            
            # DELETE 特殊处理字段 - 当ACTION为DELETE时置为空字符串
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_SALUTATION"))
            .alias("SRCC_SALUTATION"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_ENGLISHFIRSTNAME"))
            .alias("SRCC_ENGLISHFIRSTNAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_ENGLISHMIDDLENAME"))
            .alias("SRCC_ENGLISHMIDDLENAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_ENGLISHLASTNAME"))
            .alias("SRCC_ENGLISHLASTNAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_ENGLISHFULLNAME"))
            .alias("SRCC_ENGLISHFULLNAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALFIRSTNAME"))
            .alias("SRCC_LOCALFIRSTNAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALMIDDLENAME"))
            .alias("SRCC_LOCALMIDDLENAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALLASTNAME"))
            .alias("SRCC_LOCALLASTNAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALFULLNAME"))
            .alias("SRCC_LOCALFULLNAME"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALFIRSTNAME2"))
            .alias("SRCC_LOCALFIRSTNAME2"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALMIDDLENAME2"))
            .alias("SRCC_LOCALMIDDLENAME2"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALLASTNAME2"))
            .alias("SRCC_LOCALLASTNAME2"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_LOCALFULLNAME2"))
            .alias("SRCC_LOCALFULLNAME2"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_IDENTITYNUM"))
            .alias("SRCC_IDENTITYNUM"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_PASSPORTNUM"))
            .alias("SRCC_PASSPORTNUM"),
            
            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(""))
            .otherwise(F.col("ssc.SRCC_SOCIALSECURITYNUM"))
            .alias("SRCC_SOCIALSECURITYNUM"),
            
            "ssc.SRCC_GNDR_CODE",

            F.when(F.upper(F.col("ssc.SRCC_ACTION")) == "DELETE", F.lit(None))
            .otherwise(F.col("ssc.SRCC_BIRTHDAY"))
            .alias("SRCC_BIRTHDAY"),

            "ssc.SRCC_BIRTHMONTH",
            "ssc.SRCC_BIRTHYEAR",
            "ssc.SRCC_CLAS_CODE",
            "ssc.SRCC_REG_DT",
            "ssc.SRCC_REGISTRATION_TOCH_CODE",
            "ssc.SRCC_REGISTRATION_PRSN_CODE",
            "ssc.SRCC_PREFERRED_TOCH_CODE",
            "ssc.SRCC_ASSIGNED_PRSN_CODE",
            "ssc.SRCC_WLNG_CODE",
            "ssc.SRCC_SLNG_CODE",
            "ssc.SRCC_CNTR_ISOALPHA3CODE",
            "ssc.SRCC_ETHN_CODE",
            "ssc.SRCC_SKNT_CODE",
            "ssc.SRCC_HAIRT_CODE",
            "ssc.SRCC_CVLS_CODE",
            "ssc.SRCC_COMPANY",
            "ssc.SRCC_DEPARTMENT",
            "ssc.SRCC_JOBTITLE",
            "ssc.SRCC_YEARLYINCOME",
            "ssc.SRCC_DONOTCONTACT_FLAG",
            "ssc.SRCC_CURR_CODE",
            "ssc.SRCC_AGEFROM",
            "ssc.SRCC_AGETO",
            "ssc.SRCC_NATIONALITY",
            "ssc.SRCC_CHANNEL",
            "ssc.SRCC_PREFERRED_COMM_CHANNEL",
            "ssc.SRCC_ANNIVERSARY_DT",
            "ssc.SRCC_EMAILRECEIPT_FLAG",
            "ssc.SRCC_PROSPECT_FLAG",
            "ssc.SRCC_ACTIVE_FLAG",
            "ssc.SRCC_STATUS",
            F.lit("Consumer_Inter").alias("DSN"),
            "tmg.new_consumermdmkey",
            "ssc.SRCC_FIRSTPURCHASEDATE",
            "ssc.SRCC_ENGLISHNAME_QUALITY_CODE",
            "ssc.SRCC_LOCALNAME_QUALITY_CODE",
            "ssc.SRCC_LOCALNAME2_QUALITY_CODE",
            "ssc.SRCC_UNIVERSALKEY",
            "ssc.SRCC_SOURCESYSTEMCODE",
            "ssc.SRCC_MASTERCONSUMERID",
            "ssc.batch_id"
        )
        .distinct()
    )
    
    # Part 2: 处理ACS/LineBind记录（IN ACS/LineBind source codes）
    # 这部分不做DELETE处理，保留原值，也不需要timestamp比较
    consumer_part2_df = (
        itermediate_consumer_df.alias("ssc")
        .join(
            itermediate_matchgroup_df.alias("tmg"),
            (F.col("ssc.SRCC_ID") == F.col("tmg.srcc_id")) &
            (F.col("ssc.srcc_mrkt_code") == F.col("tmg.mrkt_code")),
            "inner"
        )
        .join(
            exclude_source_df.alias("es"),
            (F.col("ssc.srcc_srcs_code") == F.col("es.tmec_sourcesystemcode")) &
            (F.col("ssc.srcc_mrkt_code") == F.col("es.tmec_marketcode")),
            "inner"  # inner join只保留在ACS/LineBind中的记录
        )
        .where(
            # 所有市场统一过滤UNBIND动作
            ~(F.upper(F.col("ssc.SRCC_ACTION")) == "UNBIND")
        )
        .select(
                "ssc.SRCC_ID",
                "ssc.SRCC_ACTION",
                "ssc.SRCC_SRCS_CODE",
                "ssc.SRCC_SOURCETIMESTAMP",
                "ssc.SRCC_MRKT_CODE",
                "ssc.SRCC_AFF_CODE",
                "ssc.SRCC_DVSN_CODE",
                "ssc.SRCC_BRND_CODE",
                "ssc.SRCC_CONSUMERID",
                # Part 2不做DELETE处理，直接保留原值
                "ssc.SRCC_SALUTATION",
                "ssc.SRCC_ENGLISHFIRSTNAME",
                "ssc.SRCC_ENGLISHMIDDLENAME",
                "ssc.SRCC_ENGLISHLASTNAME",
                "ssc.SRCC_ENGLISHFULLNAME",
                "ssc.SRCC_LOCALFIRSTNAME",
                "ssc.SRCC_LOCALMIDDLENAME",
                "ssc.SRCC_LOCALLASTNAME",
                "ssc.SRCC_LOCALFULLNAME",
                "ssc.SRCC_LOCALFIRSTNAME2",
                "ssc.SRCC_LOCALMIDDLENAME2",
                "ssc.SRCC_LOCALLASTNAME2",
                "ssc.SRCC_LOCALFULLNAME2",
                "ssc.SRCC_IDENTITYNUM",
                "ssc.SRCC_PASSPORTNUM",
                "ssc.SRCC_SOCIALSECURITYNUM",
                "ssc.SRCC_GNDR_CODE",
                "ssc.SRCC_BIRTHDAY",
                "ssc.SRCC_BIRTHMONTH",
                "ssc.SRCC_BIRTHYEAR",
                "ssc.SRCC_CLAS_CODE",
                "ssc.SRCC_REG_DT",
                "ssc.SRCC_REGISTRATION_TOCH_CODE",
                "ssc.SRCC_REGISTRATION_PRSN_CODE",
                "ssc.SRCC_PREFERRED_TOCH_CODE",
                "ssc.SRCC_ASSIGNED_PRSN_CODE",
                "ssc.SRCC_WLNG_CODE",
                "ssc.SRCC_SLNG_CODE",
                "ssc.SRCC_CNTR_ISOALPHA3CODE",
                "ssc.SRCC_ETHN_CODE",
                "ssc.SRCC_SKNT_CODE",
                "ssc.SRCC_HAIRT_CODE",
                "ssc.SRCC_CVLS_CODE",
                "ssc.SRCC_COMPANY",
                "ssc.SRCC_DEPARTMENT",
                "ssc.SRCC_JOBTITLE",
                "ssc.SRCC_YEARLYINCOME",
                "ssc.SRCC_DONOTCONTACT_FLAG",
                "ssc.SRCC_CURR_CODE",
                "ssc.SRCC_AGEFROM",
                "ssc.SRCC_AGETO",
                "ssc.SRCC_NATIONALITY",
                "ssc.SRCC_CHANNEL",
                "ssc.SRCC_PREFERRED_COMM_CHANNEL",
                "ssc.SRCC_ANNIVERSARY_DT",
                "ssc.SRCC_EMAILRECEIPT_FLAG",
                "ssc.SRCC_PROSPECT_FLAG",
                "ssc.SRCC_ACTIVE_FLAG",
                "ssc.SRCC_STATUS",
                F.lit("Consumer_Inter").alias("DSN"),
                "tmg.new_consumermdmkey",
                "ssc.SRCC_FIRSTPURCHASEDATE",
                "ssc.SRCC_ENGLISHNAME_QUALITY_CODE",
                "ssc.SRCC_LOCALNAME_QUALITY_CODE",
                "ssc.SRCC_LOCALNAME2_QUALITY_CODE",
                "ssc.SRCC_UNIVERSALKEY",
                "ssc.SRCC_SOURCESYSTEMCODE",
                "ssc.SRCC_MASTERCONSUMERID",
                "ssc.batch_id"
        )
        .distinct()
    )
    
    # 合并Part 1和Part 2（UNION ALL）
    consumer_proccess_df = consumer_part1_df.unionByName(consumer_part2_df)

    # 关联t_clean_cid_group替换JPN_Rakuten&TWN_Linegift的consumerid
    consumer_proccess_df = (
        consumer_proccess_df.alias("ssc")
        .join(
            itermediate_cidgroup_df.alias("ccg"),
            (F.col("ssc.srcc_mrkt_code") == F.col("ccg.mrkt_code")) &
            (F.col("ssc.srcc_id") == F.col("ccg.srcc_id")),
            "left"
        )
        .select("ssc.*", "ccg.new_mapping_conusmer_id")
        .withColumn("SRCC_CONSUMERID", F.when(F.col("new_mapping_conusmer_id").isNotNull(), F.col("new_mapping_conusmer_id")).otherwise(F.col("SRCC_CONSUMERID")))
        .drop("new_mapping_conusmer_id")
    )

    # 去重：按照业务键去重，保留最新的timestamp记录
    # SQL: order by SRCC_SOURCETIMESTAMP desc, SRCC_ID desc
    # 注意：只有SRCC_ID和SRCC_SOURCETIMESTAMP是desc
    consumer_proccess_df = (
        consumer_proccess_df
        .withColumn("row_num", F.row_number().over(
            Window.partitionBy("srcc_srcs_code", "srcc_mrkt_code", "srcc_brnd_code", "srcc_consumerid")
            .orderBy(
                F.col("SRCC_SOURCETIMESTAMP").desc(),
                F.col("SRCC_ID").desc()
            )
        ))
        .where(F.col("row_num") == 1)
        .drop("row_num")
    )

    # 关联主表更新scon_id, creation_dt和creation_uid
    master_consumer_to_upsert = (
        consumer_proccess_df.alias("cp")
        .join(
            master_consumer_df.alias("mc"),
            (F.col("cp.srcc_srcs_code") == F.col("mc.scon_srcs_code")) &
            (F.col("cp.srcc_mrkt_code") == F.col("mc.scon_mrkt_code")) &
            (F.col("cp.srcc_brnd_code") == F.col("mc.scon_brnd_code")) &
            (F.col("cp.srcc_consumerid") == F.col("mc.scon_consumerid")),
            "left"
        )
        .select(
            [
                F.coalesce(F.col("mc.scon_id"), F.expr("uuid()")).alias("scon_id"),
                F.col("cp.new_consumermdmkey").alias("consumermdmkey"),
                F.col("cp.SRCC_ID").alias("scon_srcc_id"),
                F.col("cp.SRCC_ACTION").alias("scon_srcc_action"),
                F.col("cp.SRCC_SRCS_CODE").alias("scon_srcs_code"),
                F.col("cp.SRCC_SOURCETIMESTAMP").alias("scon_sourcetimestamp"),
                F.col("cp.SRCC_MRKT_CODE").alias("scon_mrkt_code"),
                F.col("cp.SRCC_AFF_CODE").alias("scon_aff_code"),
                F.col("cp.SRCC_DVSN_CODE").alias("scon_dvsn_code"),
                F.col("cp.SRCC_BRND_CODE").alias("scon_brnd_code"),
                F.col("cp.SRCC_CONSUMERID").alias("scon_consumerid"),
                F.col("cp.SRCC_SALUTATION").alias("scon_salutation"),
                F.col("cp.SRCC_ENGLISHFIRSTNAME").alias("scon_englishfirstname"),
                F.col("cp.SRCC_ENGLISHMIDDLENAME").alias("scon_englishmiddlename"),
                F.col("cp.SRCC_ENGLISHLASTNAME").alias("scon_englishlastname"),
                F.col("cp.SRCC_ENGLISHFULLNAME").alias("scon_englishfullname"),
                F.col("cp.SRCC_LOCALFIRSTNAME").alias("scon_localfirstname"),
                F.col("cp.SRCC_LOCALMIDDLENAME").alias("scon_localmiddlename"),
                F.col("cp.SRCC_LOCALLASTNAME").alias("scon_locallastname"),
                F.col("cp.SRCC_LOCALFULLNAME").alias("scon_localfullname"),
                F.col("cp.SRCC_LOCALFIRSTNAME2").alias("scon_localfirstname2"),
                F.col("cp.SRCC_LOCALMIDDLENAME2").alias("scon_localmiddlename2"),
                F.col("cp.SRCC_LOCALLASTNAME2").alias("scon_locallastname2"),
                F.col("cp.SRCC_LOCALFULLNAME2").alias("scon_localfullname2"),
                F.col("cp.SRCC_GNDR_CODE").alias("scon_gndr_code"),
                F.col("cp.SRCC_BIRTHDAY").alias("scon_birthday"),
                F.col("cp.SRCC_BIRTHMONTH").alias("scon_birthmonth"),
                F.col("cp.SRCC_BIRTHYEAR").alias("scon_birthyear"),
                F.col("cp.SRCC_IDENTITYNUM").alias("scon_identitynum"),
                F.col("cp.SRCC_PASSPORTNUM").alias("scon_passportnum"),
                F.col("cp.SRCC_SOCIALSECURITYNUM").alias("scon_socialsecuritynum"),
                F.col("cp.SRCC_CLAS_CODE").alias("scon_clas_code"),
                F.col("cp.SRCC_REG_DT").alias("scon_reg_dt"),
                F.col("cp.SRCC_REGISTRATION_TOCH_CODE").alias("scon_registration_toch_code"),
                F.col("cp.SRCC_REGISTRATION_PRSN_CODE").alias("scon_registration_prsn_code"),
                F.col("cp.SRCC_PREFERRED_TOCH_CODE").alias("scon_preferred_toch_code"),
                F.col("cp.SRCC_ASSIGNED_PRSN_CODE").alias("scon_assigned_prsn_code"),
                F.col("cp.SRCC_WLNG_CODE").alias("scon_wlng_code"),
                F.col("cp.SRCC_SLNG_CODE").alias("scon_slng_code"),
                F.col("cp.SRCC_CNTR_ISOALPHA3CODE").alias("scon_cntr_isoalpha3code"),
                F.col("cp.SRCC_ETHN_CODE").alias("scon_ethn_code"),
                F.col("cp.SRCC_SKNT_CODE").alias("scon_sknt_code"),
                F.col("cp.SRCC_HAIRT_CODE").alias("scon_hairt_code"),
                F.col("cp.SRCC_CVLS_CODE").alias("scon_cvls_code"),
                F.col("cp.SRCC_COMPANY").alias("scon_company"),
                F.col("cp.SRCC_DEPARTMENT").alias("scon_department"),
                F.col("cp.SRCC_JOBTITLE").alias("scon_jobtitle"),
                F.col("cp.SRCC_YEARLYINCOME").alias("scon_yearlyincome"),
                F.col("cp.SRCC_DONOTCONTACT_FLAG").alias("scon_donotcontact_flag"),
                F.col("cp.SRCC_CURR_CODE").alias("scon_curr_code"),
                F.col("cp.SRCC_AGEFROM").alias("scon_agefrom"),
                F.col("cp.SRCC_AGETO").alias("scon_ageto"),
                F.col("cp.SRCC_NATIONALITY").alias("scon_nationality"),
                F.col("cp.SRCC_CHANNEL").alias("scon_channel"),
                F.col("cp.SRCC_PREFERRED_COMM_CHANNEL").alias("scon_preferred_comm_channel"),
                F.col("cp.SRCC_ANNIVERSARY_DT").alias("scon_anniversary_dt"),
                F.col("cp.SRCC_EMAILRECEIPT_FLAG").alias("scon_emailreceipt_flag"),
                F.col("cp.SRCC_PROSPECT_FLAG").alias("scon_prospect_flag"),
                F.col("cp.SRCC_ACTIVE_FLAG").alias("scon_active_flag"),
                F.col("cp.SRCC_STATUS").alias("scon_status"),
                F.col("cp.SRCC_FIRSTPURCHASEDATE").alias("scon_firstpurchasedate"),
                F.col("cp.SRCC_ENGLISHNAME_QUALITY_CODE").alias("scon_englishname_quality_code"),
                F.col("cp.SRCC_LOCALNAME_QUALITY_CODE").alias("scon_localname_quality_code"),
                F.col("cp.SRCC_LOCALNAME2_QUALITY_CODE").alias("scon_localname2_quality_code"),
                F.lit(None).alias("SCON_COMMERCIAL_FLAG"),
                F.lit(None).alias("SCON_HRREQUESTTIMESTAMP"),
                F.lit(False).alias("SCON_DELETE_FLAG"),
                F.col("cp.SRCC_UNIVERSALKEY").alias("SCON_UNIVERSALKEY"),
                F.col("cp.SRCC_SOURCESYSTEMCODE").alias("SCON_SOURCESYSTEMCODE"),
                F.col("cp.SRCC_MASTERCONSUMERID").alias("SCON_MASTERCONSUMERID"),
                F.coalesce(F.col("mc.scon_creation_dt"), F.current_timestamp()).alias("scon_creation_dt"),
                F.coalesce(F.col("mc.scon_creation_uid"), F.lit("ELC")).alias("scon_creation_uid"),
                F.current_timestamp().alias("scon_update_dt"),
                F.lit("ELC").alias("scon_update_uid"),
                F.lit(False).alias("scon_cbr_flag"),
                F.col("cp.batch_id"),
                F.lit(task_id).alias("task_id")
            ]
        )
        .cache()
    )
    
    print(f'master consumer to-upsert count: {master_consumer_to_upsert.count()}')

    # 4. 使用DeltaTable的merge操作实现更新和插入    
    master_consumer_delta_table = DeltaTable.forName(spark, master_consumer_table_name)
    master_consumer_merge_result = (
        master_consumer_delta_table
        .alias("target")
        .merge(
            master_consumer_to_upsert.alias("source"),
            "target.scon_srcs_code = source.scon_srcs_code AND "
            "target.scon_mrkt_code = source.scon_mrkt_code AND "
            "target.scon_brnd_code = source.scon_brnd_code AND "
            "target.scon_consumerid = source.scon_consumerid"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    master_consumer_to_upsert.unpersist()
    # merge_result = master_consumer_merge_result.first()
    # print(f'master consumer inserted count: {merge_result.num_inserted_rows}, updated count: {merge_result.num_updated_rows}')

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("5.1_generate_master_consumer_tables", "05-1", "consumerlist", task_id=task_id) as logger:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")

    # 更新srcc_action为DELETE的记录，清空Pii
    update_delete_action_records_with_clearpii(task_id)

    # 更新ACS数据task_id，便于数据下发（标记需要重新进行CBR处理的记录）
    # 逻辑说明：查找所有在ACS系统（或其他排除系统）中存在的emedia/phone联系方式，这些联系方式在主数据中也存在，需要标记为重新处理(task_id)
    update_acs_optin_task_id(task_id)

    # 常规master consumer表的Upsert逻辑
    calc_consumer_master(task_id)